In [1]:
from typing import Any, Dict

import hydra
import numpy as np
import omegaconf
import torch
import pytorch_lightning as pl
import torch.nn as nn
from torch.nn import functional as F
from torch_scatter import scatter
from tqdm import tqdm

from cdvae.common.utils import PROJECT_ROOT
from cdvae.common.data_utils import (
    EPSILON, cart_to_frac_coords, mard, lengths_angles_to_volume,
    frac_to_cart_coords, min_distance_sqr_pbc)
from cdvae.pl_modules.embeddings import MAX_ATOMIC_NUM
from cdvae.pl_modules.embeddings import KHOT_EMBEDDINGS

#added by Tsach
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator
#import Batch
from torch_geometric.data import Batch
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)
torch.set_printoptions(threshold=50000) # use this if you want to print the entire tensor


In [2]:
 import time
import argparse
import torch

from tqdm import tqdm
from torch.optim import Adam
from pathlib import Path
from types import SimpleNamespace
from torch_geometric.data import Batch
from torch_geometric.data.separate import separate

#import a library that allows you to reload a module
from importlib import reload

from eval_utils import load_model

all_frac_coords_stack = []
all_atom_types_stack = []
frac_coords = []
num_atoms = []
atom_types = []
lengths = []
angles = []
input_data_list = []

#my code 
list_of_idxs = []
list_of_batchs = []

In [4]:
model_path = Path("/home/gridsan/tmackey/hydra/singlerun/2023-10-20/non_conditional_cdvae_predictPXRD")
model, test_loader, cfg = load_model(model_path, True)

loader = test_loader

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/hydra/experimental/compose.py:16: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  warnings.warn(


RuntimeError: Error(s) in loading state_dict for CDVAE:
	Missing key(s) in state_dict: "fc_xrd_loc.0.weight", "fc_xrd_loc.0.bias", "fc_xrd_loc.2.weight", "fc_xrd_loc.2.bias", "fc_xrd_int.0.weight", "fc_xrd_int.0.bias", "fc_xrd_int.2.weight", "fc_xrd_int.2.bias". 
	Unexpected key(s) in state_dict: "fc_diffraction_pattern.0.weight", "fc_diffraction_pattern.0.bias", "fc_diffraction_pattern.2.weight", "fc_diffraction_pattern.2.bias", "fc_diffraction_pattern.4.weight", "fc_diffraction_pattern.4.bias", "fc_diffraction_pattern.6.weight", "fc_diffraction_pattern.6.bias", "fc_diffraction_pattern.8.weight", "fc_diffraction_pattern.8.bias", "atomic_and_diffraction_encoder.4.weight", "atomic_and_diffraction_encoder.4.bias", "prior_encoder.4.weight", "prior_encoder.4.bias", "prior_encoder.6.weight", "prior_encoder.6.bias", "prior_encoder.8.weight", "prior_encoder.8.bias", "prior_encoder.10.weight", "prior_encoder.10.bias", "prior_encoder.12.weight", "prior_encoder.12.bias", "post_encoder.4.weight",